<a href="https://colab.research.google.com/github/gauribahl/ai-resume-optimizer/blob/main/AI_Resume_Optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install gradio -q
!pip install gradio google-genai

In [12]:
# Cell 2: Mount Google Drive and create folder structure
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/AI_Resume_Optimizer/resumes"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"✅ Google Drive mounted.")
print(f"✅ Save directory ready: {SAVE_DIR}")

Mounted at /content/drive
✅ Google Drive mounted.
✅ Save directory ready: /content/drive/MyDrive/AI_Resume_Optimizer/resumes


In [17]:
# Cell 3: Full AI Resume Optimizer App with History
!pip install PyMuPDF -q

import gradio as gr
from google import genai
import fitz  # PyMuPDF
import os
import json
from datetime import datetime

# ── Config ────────────────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/MyDrive/AI_Resume_Optimizer/resumes"
HISTORY_FILE = "/content/drive/MyDrive/AI_Resume_Optimizer/history.json"
JOB_CATEGORIES = ["ML Engineer", "Data Scientist", "AI Engineer", "Computer Vision", "Software Engineer"]

api_key_store = {"key": None}
session_state = {"jd_analysis": "", "questions": [], "last_job_title": "Unknown"}


# ── History Helpers ───────────────────────────────────────────────────────────
def load_history_data():
    if not os.path.exists(HISTORY_FILE):
        return []
    try:
        with open(HISTORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return []


def save_history_entry(job_title, category, optimized_text):
    history = load_history_data()
    entry = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "job_title": job_title,
        "category": category,
        "preview": optimized_text.strip()[:100],
    }
    history.append(entry)
    os.makedirs(os.path.dirname(HISTORY_FILE), exist_ok=True)
    with open(HISTORY_FILE, "w", encoding="utf-8") as f:
        json.dump(history, f, indent=2, ensure_ascii=False)


def format_history_display():
    history = load_history_data()
    if not history:
        return "📭 No optimization history yet. Optimize a resume to start logging."

    lines = []
    for i, entry in enumerate(reversed(history), 1):
        lines.append(f"{'='*60}")
        lines.append(f"#{i}  🕐 {entry.get('timestamp', 'N/A')}")
        lines.append(f"💼  Job Title : {entry.get('job_title', 'Unknown')}")
        lines.append(f"📁  Category  : {entry.get('category', 'Not saved')}")
        lines.append(f"📝  Preview   : {entry.get('preview', '')}...")
        lines.append("")
    return "\n".join(lines)


def extract_job_title_from_analysis(jd_analysis):
    for line in jd_analysis.splitlines():
        lower = line.lower()
        if "job title" in lower or "position" in lower or "role" in lower:
            if ":" in line:
                return line.split(":", 1)[1].strip()
            return line.strip()
    return "Unknown"


# ── Helpers ───────────────────────────────────────────────────────────────────
def set_api_key(key):
    key = key.strip()
    if not key:
        return "⚠️ Please enter a valid API key."
    api_key_store["key"] = key
    return "✅ API key set successfully!"


def extract_pdf_text(file_obj):
    if file_obj is None:
        return ""
    try:
        doc = fitz.open(file_obj.name)
        text_parts = [page.get_text() for page in doc]
        doc.close()
        extracted = "\n".join(text_parts).strip()
        if not extracted:
            return "⚠️ No text could be extracted from this PDF. It may be image-based — try copy-pasting your resume text manually."
        return extracted
    except Exception as e:
        return f"❌ Error reading PDF: {str(e)}"


def handle_pdf_upload(file_obj):
    if file_obj is None:
        return "", "No file uploaded yet.", "0 characters"
    extracted = extract_pdf_text(file_obj)
    if extracted.startswith("⚠️") or extracted.startswith("❌"):
        return "", extracted, "0 characters"
    page_count = fitz.open(file_obj.name).page_count
    status = f"✅ Extracted {len(extracted):,} characters from {page_count} page(s). Review and edit below."
    char_count = f"{len(extracted):,} characters"
    return extracted, status, char_count


# ── Character count helpers ───────────────────────────────────────────────────
def count_resume_chars(text):
    return f"{len(text):,} characters"

def count_jd_chars(text):
    return f"{len(text):,} characters"

def count_output_chars(text):
    return f"{len(text):,} characters"


# ── Step 1: Analyze JD + Generate Clarifying Questions ───────────────────────
def analyze_and_question(resume, job_description):
    if not api_key_store["key"]:
        return (
            "⚠️ Please set your Gemini API key first.",
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False),
        )

    missing = []
    if not resume.strip():
        missing.append("resume")
    if not job_description.strip():
        missing.append("job description")
    if missing:
        return (
            f"⚠️ Please provide your {' and '.join(missing)} before analyzing.",
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False),
        )

    try:
        client = genai.Client(api_key=api_key_store["key"])

        # Call 1: Analyze JD — request structured markdown output
        analysis_prompt = f"""Analyze this job description and return a structured markdown summary with the following sections:

## 🏢 Job Overview
- **Job Title**: ...
- **Company**: ...

## ✅ Must-Have Skills
- List each required skill as a bullet point

## 💡 Nice-to-Have Skills
- List each preferred skill as a bullet point

## 📋 Key Responsibilities
- List the top 4-6 responsibilities as bullet points

## 🔑 ATS Keywords
- List the most important ATS/SEO keywords as a comma-separated inline list

Job Description:
{job_description}"""

        analysis_response = client.models.generate_content(
            model="gemini-2.5-flash", contents=analysis_prompt
        )
        jd_analysis = analysis_response.text
        session_state["jd_analysis"] = jd_analysis
        session_state["last_job_title"] = extract_job_title_from_analysis(jd_analysis)

        # Call 2: Generate clarifying questions
        questions_prompt = f"""You are a professional resume coach. Compare this resume against the job description analysis below.

Identify 2-3 gaps or opportunities where the candidate MIGHT have relevant experience that isn't captured in their resume. Generate specific, targeted questions to uncover that experience.

Rules:
- Ask only about things mentioned in the JD but unclear or missing from the resume
- Make questions specific (reference actual JD requirements)
- Do NOT ask about things clearly present in the resume
- Output ONLY a JSON array of question strings, no other text
- Example format: ["Question 1?", "Question 2?", "Question 3?"]

--- RESUME ---
{resume}

--- JD ANALYSIS ---
{jd_analysis}

--- FULL JOB DESCRIPTION ---
{job_description}"""

        q_response = client.models.generate_content(
            model="gemini-2.5-flash", contents=questions_prompt
        )

        raw = q_response.text.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        questions = json.loads(raw.strip())
        questions = questions[:3]
        session_state["questions"] = questions

        q_updates = []
        for i in range(3):
            if i < len(questions):
                q_updates.append(gr.update(visible=True, label=f"Q{i+1}: {questions[i]}", value=""))
            else:
                q_updates.append(gr.update(visible=False, value=""))

        status_msg = "✅ JD analyzed. Answer the question(s) below, then click **Optimize My Resume**."

        return (
            status_msg,
            q_updates[0], q_updates[1], q_updates[2],
            gr.update(visible=True),
            gr.update(visible=True),
            gr.update(value=jd_analysis),
            gr.update(visible=True),
        )

    except json.JSONDecodeError:
        fallback = [
            "Can you describe any relevant projects or experience that may not be fully captured in your resume?",
            "Are there any specific tools, technologies, or methodologies from the JD that you've used but haven't listed?",
        ]
        session_state["questions"] = fallback
        q_updates = [
            gr.update(visible=True, label=f"Q{i+1}: {fallback[i]}", value="") if i < len(fallback)
            else gr.update(visible=False, value="")
            for i in range(3)
        ]
        return (
            "✅ JD analyzed (used fallback questions). Answer below, then click **Optimize My Resume**.",
            q_updates[0], q_updates[1], q_updates[2],
            gr.update(visible=True),
            gr.update(visible=True),
            gr.update(value=""),
            gr.update(visible=True),
        )

    except Exception as e:
        return (
            f"❌ Error: {str(e)}",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False),
            gr.update(visible=False), gr.update(visible=False),
            gr.update(value=""), gr.update(visible=True),
        )


# ── Step 2: Optimize Resume using answers ────────────────────────────────────
def optimize_resume(resume, job_description, ans1, ans2, ans3, category):
    if not api_key_store["key"]:
        return "⚠️ Please set your Gemini API key first.", "0 characters"
    if not resume.strip() or not job_description.strip():
        return "⚠️ Resume and job description are required.", "0 characters"

    jd_analysis = session_state.get("jd_analysis", "")
    questions = session_state.get("questions", [])
    job_title = session_state.get("last_job_title", "Unknown")

    answers = [ans1, ans2, ans3]
    qa_context_parts = []
    for i, q in enumerate(questions):
        a = answers[i].strip() if i < len(answers) else ""
        if a:
            qa_context_parts.append(f"Q: {q}\nA: {a}")
    qa_context = "\n\n".join(qa_context_parts) if qa_context_parts else "No additional context provided."

    try:
        client = genai.Client(api_key=api_key_store["key"])

        optimize_prompt = f"""You are a professional resume writer. Rewrite this resume to be perfectly tailored to the target job.

Rules:
* Naturally weave in keywords from the job description
* Rewrite the professional summary to match the target role
* Emphasize relevant skills and achievements
* IMPORTANT: Only use information from the original resume AND the candidate's answers below. Do not fabricate anything.
* Incorporate the candidate's clarifying answers to add missing but relevant experience/context
* Use strong action verbs and quantify where possible
* Output the COMPLETE rewritten resume as clean text

--- ORIGINAL RESUME ---
{resume}

--- JOB DESCRIPTION ANALYSIS ---
{jd_analysis}

--- FULL JOB DESCRIPTION ---
{job_description}

--- CANDIDATE'S CLARIFYING ANSWERS (incorporate this additional context) ---
{qa_context}"""

        response = client.models.generate_content(
            model="gemini-2.5-flash", contents=optimize_prompt
        )
        optimized_text = response.text

        try:
            save_history_entry(
                job_title=job_title,
                category=category,
                optimized_text=optimized_text,
            )
        except Exception as hist_err:
            print(f"[History] Failed to save entry: {hist_err}")

        return optimized_text, f"{len(optimized_text):,} characters"

    except Exception as e:
        return f"❌ Error calling Gemini API: {str(e)}", "0 characters"


# ── Save / Library ────────────────────────────────────────────────────────────
def save_resume(optimized_text, category):
    if not optimized_text.strip():
        return "⚠️ Nothing to save — optimize a resume first."
    category_safe = category.replace(" ", "_")
    folder = os.path.join(SAVE_DIR, category_safe)
    os.makedirs(folder, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{timestamp}_{category_safe}.txt"
    filepath = os.path.join(folder, filename)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"Category: {category}\n")
        f.write(f"Saved: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("=" * 60 + "\n\n")
        f.write(optimized_text)
    return f"✅ Saved as **{filename}** in `{category}` folder."


def load_library():
    if not os.path.exists(SAVE_DIR):
        return "⚠️ Save directory not found. Make sure Google Drive is mounted (Cell 1)."
    output_parts = []
    found_any = False
    for category in sorted(os.listdir(SAVE_DIR)):
        cat_path = os.path.join(SAVE_DIR, category)
        if not os.path.isdir(cat_path):
            continue
        files = sorted([f for f in os.listdir(cat_path) if f.endswith(".txt")], reverse=True)
        if not files:
            continue
        found_any = True
        display_name = category.replace("_", " ")
        output_parts.append(f"{'='*60}")
        output_parts.append(f"📁  {display_name}  ({len(files)} resume{'s' if len(files)!=1 else ''})")
        output_parts.append(f"{'='*60}")
        for fname in files:
            fpath = os.path.join(cat_path, fname)
            output_parts.append(f"\n── {fname} ──")
            with open(fpath, "r", encoding="utf-8") as f:
                content = f.read()
            lines = content.split("\n")
            header = "\n".join(lines[:3])
            body_preview = "\n".join(lines[4:])[:300]
            output_parts.append(header)
            output_parts.append(body_preview + (" ..." if len("\n".join(lines[4:])) > 300 else ""))
        output_parts.append("")
    if not found_any:
        return "📭 No saved resumes yet. Optimize and save a resume to see it here."
    return "\n".join(output_parts)


# ── Copy to Clipboard JS ──────────────────────────────────────────────────────
COPY_JS = """
async () => {
    const textareas = document.querySelectorAll('textarea');
    let outputTextarea = null;
    for (const ta of textareas) {
        const label = ta.closest('.form-group, [class*="block"]')?.querySelector('label, span');
        if (label && label.textContent.includes('Optimized Resume')) {
            outputTextarea = ta;
            break;
        }
    }
    // Fallback: grab the last visible textarea (the output box)
    if (!outputTextarea) {
        const allTa = Array.from(textareas).filter(t => t.offsetParent !== null);
        outputTextarea = allTa[allTa.length - 1];
    }
    if (outputTextarea && outputTextarea.value) {
        await navigator.clipboard.writeText(outputTextarea.value);
        const btn = document.activeElement;
        const original = btn.textContent;
        btn.textContent = '✅ Copied!';
        setTimeout(() => { btn.textContent = original; }, 2000);
    }
}
"""


# ── UI ────────────────────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.Markdown("""
# 🤖 AI Resume Optimizer
**Tailor your resume to any job description in seconds — powered by Gemini AI.**

Upload your resume and paste a job description. The AI will analyze the role, ask clarifying questions to surface hidden experience, then rewrite your resume with the right keywords and framing to pass ATS filters and impress hiring managers.
""")

    # ── Settings ──────────────────────────────────────────────────────────────
    with gr.Accordion("⚙️ Settings", open=True):
        with gr.Row():
            api_key_input = gr.Textbox(
                label="Gemini API Key", placeholder="Paste your Gemini API key here...",
                type="password", scale=4,
            )
            set_key_btn = gr.Button("Set API Key", variant="secondary", scale=1)
        api_key_status = gr.Textbox(
            label="Status", value="⚠️ API key not set.", interactive=False, lines=1
        )
    set_key_btn.click(fn=set_api_key, inputs=[api_key_input], outputs=[api_key_status])

    gr.Markdown("---")

    with gr.Tabs():
        # ── Tab 1: Optimizer ──────────────────────────────────────────────────
        with gr.Tab("✏️ Optimizer"):

            with gr.Accordion("📄 Upload Resume PDF (optional)", open=True):
                gr.Markdown("Upload a PDF to auto-fill the resume text box below.")
                with gr.Row():
                    pdf_upload = gr.File(label="Upload PDF", file_types=[".pdf"], scale=3)
                    pdf_status = gr.Textbox(
                        label="Extraction Status", value="No file uploaded yet.",
                        interactive=False, lines=1, scale=2,
                    )

            gr.Markdown("---")

            with gr.Row():
                with gr.Column():
                    resume_input = gr.Textbox(
                        label="Your Resume (paste or auto-filled from PDF above)",
                        lines=12, placeholder="Paste your resume here, or upload a PDF above to auto-fill...",
                    )
                    resume_char_count = gr.Markdown("0 characters")

                with gr.Column():
                    jd_input = gr.Textbox(
                        label="Job Description",
                        lines=12, placeholder="Paste the job description here...",
                    )
                    jd_char_count = gr.Markdown("0 characters")

            # Live character counts
            resume_input.change(fn=count_resume_chars, inputs=[resume_input], outputs=[resume_char_count])
            jd_input.change(fn=count_jd_chars, inputs=[jd_input], outputs=[jd_char_count])

            analyze_btn = gr.Button("🔍 Analyze & Generate Questions", variant="primary", size="lg")

            with gr.Row(visible=False) as analysis_status_row:
                analysis_status = gr.Markdown("")

            # JD Analysis display (markdown rendered)
            with gr.Accordion("📊 JD Analysis", open=True, visible=False) as jd_analysis_accordion:
                jd_analysis_display = gr.Markdown("")

            with gr.Column(visible=False) as questions_panel:
                gr.Markdown("### 💬 Answer These Questions to Improve Your Resume")
                gr.Markdown("The AI identified potential gaps. Your answers will be woven into the optimized resume.")
                q1_box = gr.Textbox(label="", lines=3, visible=False, placeholder="Your answer...")
                q2_box = gr.Textbox(label="", lines=3, visible=False, placeholder="Your answer...")
                q3_box = gr.Textbox(label="", lines=3, visible=False, placeholder="Your answer...")

            optimize_btn = gr.Button(
                "⚡ Optimize My Resume", variant="primary", size="lg", visible=False
            )

            gr.Markdown("### 📄 Optimized Resume")
            with gr.Row():
                copy_btn = gr.Button("📋 Copy to Clipboard", variant="secondary", scale=1)
            output = gr.Textbox(
                label="Optimized Resume",
                lines=20,
                show_label=False,
                placeholder="Your optimized resume will appear here after clicking 'Optimize My Resume'...",
            )
            output_char_count = gr.Markdown("0 characters")

            copy_btn.click(fn=None, js=COPY_JS)

            gr.Markdown("### 💾 Save to Google Drive")
            with gr.Row():
                category_dropdown = gr.Dropdown(
                    choices=JOB_CATEGORIES, value=JOB_CATEGORIES[0],
                    label="Job Category", scale=3,
                )
                save_btn = gr.Button("💾 Save Resume", variant="secondary", scale=1)
            save_status = gr.Markdown("")

            # ── Events ────────────────────────────────────────────────────────
            pdf_upload.change(
                fn=handle_pdf_upload,
                inputs=[pdf_upload],
                outputs=[resume_input, pdf_status, resume_char_count],
            )

            analyze_btn.click(
                fn=analyze_and_question,
                inputs=[resume_input, jd_input],
                outputs=[
                    analysis_status,
                    q1_box, q2_box, q3_box,
                    questions_panel,
                    optimize_btn,
                    jd_analysis_display,
                    analysis_status_row,
                ],
            ).then(
                fn=lambda: gr.update(visible=True),
                inputs=[],
                outputs=[jd_analysis_accordion],
            )

            optimize_btn.click(
                fn=optimize_resume,
                inputs=[resume_input, jd_input, q1_box, q2_box, q3_box, category_dropdown],
                outputs=[output, output_char_count],
            )

            output.change(fn=count_output_chars, inputs=[output], outputs=[output_char_count])

            save_btn.click(
                fn=save_resume,
                inputs=[output, category_dropdown],
                outputs=[save_status],
            )

        # ── Tab 2: Resume Library ─────────────────────────────────────────────
        with gr.Tab("📚 Resume Library"):
            gr.Markdown("Browse all your saved resumes, grouped by job category.")
            load_btn = gr.Button("🔄 Load Saved Resumes", variant="primary")
            library_output = gr.Textbox(
                label="Saved Resumes", lines=30, interactive=False,
                placeholder="Click 'Load Saved Resumes' to view your library...",
            )
            load_btn.click(fn=load_library, inputs=[], outputs=[library_output])

        # ── Tab 3: History ────────────────────────────────────────────────────
        with gr.Tab("🕐 History"):
            gr.Markdown("### Optimization History")
            gr.Markdown(
                "Every time you optimize a resume, an entry is automatically logged here "
                f"and saved to `{HISTORY_FILE}` on your Google Drive."
            )
            load_history_btn = gr.Button("🔄 Load History", variant="primary")
            history_entry_count = gr.Markdown("")
            history_output = gr.Textbox(
                label="Past Optimizations", lines=30, interactive=False,
                placeholder="Click 'Load History' to view past optimizations...",
            )

            def load_history_ui():
                history = load_history_data()
                count_msg = f"**{len(history)} optimization(s) logged total.**"
                display = format_history_display()
                return count_msg, display

            load_history_btn.click(
                fn=load_history_ui,
                inputs=[],
                outputs=[history_entry_count, history_output],
            )

demo.launch(share=True)

/tmp/ipython-input-264/2782698000.py:401: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo")) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1ede5f228966dda255.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
